# 🏥 Health & Lifestyle Analysis: The Complete EDA Pipeline

**Author:** Elite Data Scientist Pipeline  
**Purpose:** Production-ready exploratory data analysis with deep statistical rigor  
**Dataset:** Early Wakeup Health & Lifestyle Dataset

---

## 📋 Pipeline Philosophy

EDA is not "plotting pretty charts." It is a **diagnostic process** where every statistical test,  
every visualization, and every transformation decision is grounded in the data's properties.  
This notebook follows the complete workflow:

```
Phase 0: Environment Setup → Phase 1: Data Quality → Phase 2: Univariate Analysis  
→ Phase 3: Bivariate/Multivariate → Phase 4: Feature Engineering → Phase 5: Validation
```



---

# Phase 0: Environment Setup & Configuration

## Why This Matters
Before touching data, we establish:
- **Reproducibility:** Fixed random seeds, documented package versions
- **Visual consistency:** Custom color palette that is colorblind-friendly and publication-ready
- **Computational efficiency:** GPU detection for downstream deep learning

## Custom Aesthetic Configuration
We use a **cool-toned palette** (`#E3FDFD` to `#71C9CE`) because:
1. It reduces visual fatigue during long analysis sessions
2. Cool colors are perceived as "clinical" and "trustworthy" — fitting for health data
3. High contrast with white data points ensures readability in dark-themed plots



In [ ]:
# CELL 1: IMPORTS & CONFIGURATION
# ═══════════════════════════════════════════════════════════════════════════════

# ─── Core Data Science Libraries ───
import numpy as np              # Numerical computing: arrays, linear algebra, random sampling
import pandas as pd             # Data manipulation: DataFrames, Series, I/O operations
import math                     # Mathematical functions: ceil, floor, sqrt, etc.

# ─── Visualization Libraries ───
import matplotlib.pyplot as plt # Low-level plotting: figures, axes, custom plots
import matplotlib.colors as mcolors  # Color manipulation: palettes, normalization
import seaborn as sns           # High-level statistical visualization: heatmaps, distributions

# ─── Statistical Analysis ───
from scipy.stats import trim_mean      # Robust mean: removes extreme % from both ends
from scipy import stats               # Comprehensive statistics: distributions, tests, Q-Q plots
from scipy.stats import mannwhitneyu, kruskal  # Non-parametric tests (explained in Phase 3)

# ─── Feature Engineering & Preprocessing ───
from sklearn.preprocessing import (
    PowerTransformer,     # Automated skewness reduction (Yeo-Johnson / Box-Cox)
    RobustScaler,         # Scaling using median & IQR (outlier-immune)
    StandardScaler,       # Z-score normalization (mean=0, std=1)
    MinMaxScaler          # Min-max scaling to [0,1] range
)
from sklearn.impute import SimpleImputer  # Systematic missing value imputation

# ─── Deep Learning & Optimization ───
import torch as t               # PyTorch: neural network framework
import torch.nn as nn           # Neural network layers and architectures
import optuna as ot             # Hyperparameter optimization framework
import wandb                    # Experiment tracking and visualization

# ─── NLP / Semantic Encoding ───
# sentence_transformers: Converts categorical text into dense semantic vectors (384-dim)
# This is superior to one-hot encoding because it captures MEANING, not just identity
from sentence_transformers import SentenceTransformer

# ─── System & I/O ───
import os
import warnings
warnings.filterwarnings('ignore')  # Suppress non-critical warnings for clean output

# ═══════════════════════════════════════════════════════════════════════════════
# REPRODUCIBILITY: Fix random seeds across all libraries
# ═══════════════════════════════════════════════════════════════════════════════
# Why? Random processes (train/test splits, neural network initialization, 
# stochastic optimization) must be reproducible for scientific validity.
SEED = 42
np.random.seed(SEED)
t.manual_seed(SEED)
if t.cuda.is_available():
    t.cuda.manual_seed_all(SEED)

# ═══════════════════════════════════════════════════════════════════════════════
# HARDWARE DETECTION: GPU acceleration for downstream neural network training
# ═══════════════════════════════════════════════════════════════════════════════
# CUDA (Compute Unified Device Architecture) enables parallel computation on NVIDIA GPUs.
# For tabular data with embeddings, GPU acceleration can provide 10-50x speedup.
device = "cuda" if t.cuda.is_available() else "cpu"
print(f"🖥️  Accelerating with: {device.upper()}")
if device == "cuda":
    print(f"   GPU: {t.cuda.get_device_name(0)}")
    print(f"   Memory: {t.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ═══════════════════════════════════════════════════════════════════════════════
# CUSTOM AESTHETICS: Publication-Ready Visualization Theme
# ═══════════════════════════════════════════════════════════════════════════════
# Palette rationale:
#   #E3FDFD (Lightest) → Background color, reduces eye strain
#   #CBF1F5           → Grid lines, subtle separators
#   #A6E3E9           → Secondary elements, annotations
#   #71C9CE (Darkest) → Primary accents, borders, emphasis
#
# This is a monochromatic teal scheme. Monochromatic palettes are:
#   - Colorblind-safe (no red-green confusion)
#   - Professional and calming (appropriate for health data)
#   - High contrast when paired with white data points

custom_palette = ['#E3FDFD', '#CBF1F5', '#A6E3E9', '#71C9CE']

sns.set_palette(sns.color_palette(custom_palette))
sns.set_theme(
    style='darkgrid',
    rc={
        'axes.facecolor': '#E3FDFD',      # Plot background
        'figure.facecolor': '#E3FDFD',    # Figure background
        'axes.edgecolor': '#71C9CE',      # Axis border color
        'grid.color': '#CBF1F5',          # Grid line color
        'text.color': 'black',            # All text color
        'axes.labelcolor': 'black',       # Axis label color
        'xtick.color': 'black',           # X-tick label color
        'ytick.color': 'black',           # Y-tick label color
        'axes.titlecolor': 'black',       # Title color
        'figure.titlesize': 16,           # Figure title size
        'axes.titlesize': 14,             # Axes title size
        'axes.labelsize': 12,             # Axis label size
    }
)

# Global plot constants for consistency
BG_COLOR = "#E3FDFD"      # Background color reference
TEXT_COLOR = "black"      # Text color reference
ACCENT_COLOR = "#71C9CE"  # Primary accent color
GRID_COLOR = "#CBF1F5"    # Grid line color

print("✅ Environment configured. All systems ready.")



---

# Phase 1: Data Loading & Quality Assessment

## What We're Doing Here
Before any analysis, we must understand:
1. **Structure:** How many rows, columns, what do they represent?
2. **Types:** Are numeric columns truly numeric? Are dates parsed correctly?
3. **Quality:** Missing values, duplicates, impossible values?
4. **Memory:** Is the dataset efficiently stored?

## Why This Phase Is Non-Negotiable
Skipping data quality checks is like building a house on quicksand.  
Every downstream analysis — every statistical test, every model — assumes the data is valid.  
Garbage in, garbage out. Always.



In [ ]:
# CELL 2: DATA LOADING
# ═══════════════════════════════════════════════════════════════════════════════

# Load the raw dataset
# The dataset contains health and lifestyle metrics for individuals with varying wake-up habits.
Raw_Data = pd.read_csv('/kaggle/input/datasets/nalisha/early-wakeup-health-and-lifestyle-dataset/early_wakeup_health_dataset.csv')
print("📊 Dataset loaded successfully.")

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3: STRUCTURAL OVERVIEW
# ═══════════════════════════════════════════════════════════════════════════════

print(f"{'═'*60}")
print("STRUCTURAL OVERVIEW")
print(f"{'═'*60}")
print(f"Shape (rows, columns): {Raw_Data.shape}")
print(f"Total cells: {Raw_Data.shape[0] * Raw_Data.shape[1]:,}")
print(f"Memory usage: {Raw_Data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"{'─'*60}")
print("COLUMN NAMES & DATA TYPES:")
print(f"{'─'*60}")
for col, dtype in zip(Raw_Data.columns, Raw_Data.dtypes):
    print(f"  {col:30s} → {str(dtype):15s}")



In [ ]:
# CELL 4: INITIAL STATISTICAL SNAPSHOT
# ═══════════════════════════════════════════════════════════════════════════════

print(f"
{'═'*60}")
print("STATISTICAL SNAPSHOT (Numeric Columns)")
print(f"{'═'*60}")
# describe() gives: count, mean, std, min, 25%, 50%, 75%, max
# This is our FIRST diagnostic tool. We compare mean vs median to detect skewness.
desc = Raw_Data.describe()
print(desc.T.round(2))

print(f"
{'═'*60}")
print("CATEGORICAL COLUMNS OVERVIEW")
print(f"{'═'*60}")
for col in Raw_Data.select_dtypes(include=['object']).columns:
    print(f"
  {col}:")
    print(f"    Unique values: {Raw_Data[col].nunique()}")
    print(f"    Top 5: {Raw_Data[col].value_counts().head(5).to_dict()}")



In [ ]:
# CELL 5: DATA QUALITY DIAGNOSTICS
# ═══════════════════════════════════════════════════════════════════════════════

print(f"
{'═'*60}")
print("DATA QUALITY REPORT")
print(f"{'═'*60}")

# ─── Duplicate Analysis ───
# Duplicates can cause data leakage (same row in train and test) or inflate metrics.
duplicates = Raw_Data.duplicated().sum()
print(f"
🔍 Duplicates: {duplicates}")
if duplicates > 0:
    print(f"   ⚠️  WARNING: {duplicates} duplicate rows detected. Remove before splitting.")

# ─── Missing Value Analysis ───
# Missing data is not just "empty cells" — it's information.
# The PATTERN of missingness (random vs systematic) determines our strategy.
missing = Raw_Data.isnull().sum()
missing_pct = (missing / len(Raw_Data) * 100).round(2)
missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct,
    'Data Type': Raw_Data.dtypes
})
missing_report = missing_report[missing_report['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print(f"
🔍 Missing Values:")
if len(missing_report) > 0:
    print(missing_report)
    print(f"
   ⚠️  Total missing cells: {missing.sum():,} ({missing.sum() / (Raw_Data.shape[0]*Raw_Data.shape[1])*100:.2f}% of all data)")
else:
    print("   ✅ No missing values detected.")

# ─── Memory Optimization Check ───
print(f"
💾 Memory Usage by Column:")
mem_usage = Raw_Data.memory_usage(deep=True)
for col, mem in mem_usage.items():
    if col != 'Index':
        print(f"   {col:30s}: {mem/1024:8.2f} KB")



---

## 🧹 Initial Data Cleaning

### Why We Drop These Columns

| Column | Reason for Dropping |
|--------|---------------------|
| `Person_ID` | Pure identifier, no predictive value. Including it would cause the model to memorize IDs rather than learn patterns. |
| `Height_cm` | Redundant with BMI (which incorporates height and weight). Multicollinearity risk. |
| `Weight_kg` | Same as above — captured by BMI. |
| `Early_Waker` | This is likely derived from `Wake_Up_Time`. Using derived features alongside their source creates redundancy. |

### Time Feature Engineering
Raw time strings (`"06:30"`) are useless for models. We convert to **minutes since midnight**:
- `"06:30"` → $6 \times 60 + 30 = 390$ minutes
- This creates a continuous numeric feature that preserves the cyclic nature of time
- Alternative: Cyclical encoding $(sin(2\pi \times t/1440), cos(2\pi \times t/1440))$ for neural networks



In [ ]:
# CELL 6: INITIAL CLEANING & TYPE CONVERSION
# ═══════════════════════════════════════════════════════════════════════════════

# Create a working copy — NEVER modify raw data directly
Data = Raw_Data.copy()

# ─── Drop redundant/identifier columns ───
columns_to_drop = ['Height_cm', 'Weight_kg', 'Early_Waker', 'Person_ID']
Data = Data.drop(columns=columns_to_drop, axis=1)
print(f"✅ Dropped columns: {columns_to_drop}")

# ─── Convert time strings to minutes since midnight ───
# WHY: Machine learning models require numeric inputs. Time strings are categorical
# and would lose the continuous relationship (23:59 is close to 00:00, not far).
# Minutes since midnight preserves ordinality and enables meaningful distances.
Data['Wake_Up_Time'] = pd.to_datetime(Data['Wake_Up_Time'], format='%H:%M')
Data['Sleep_Time'] = pd.to_datetime(Data['Sleep_Time'], format='%H:%M')

Data['Wake_Up_Time'] = Data['Wake_Up_Time'].dt.hour * 60 + Data['Wake_Up_Time'].dt.minute
Data['Sleep_Time'] = Data['Sleep_Time'].dt.hour * 60 + Data['Sleep_Time'].dt.minute

print(f"✅ Time features converted to minutes since midnight")
print(f"   Wake_Up_Time range: {Data['Wake_Up_Time'].min()} - {Data['Wake_Up_Time'].max()} minutes")
print(f"   Sleep_Time range: {Data['Sleep_Time'].min()} - {Data['Sleep_Time'].max()} minutes")

# ─── Separate column types ───
# This separation drives all downstream analysis and preprocessing decisions.
cols_categorical = Data.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = Data.select_dtypes(include=["number"]).columns.tolist()

print(f"
📊 Column Classification:")
print(f"   Numeric ({len(numeric_cols)}): {numeric_cols}")
print(f"   Categorical ({len(cols_categorical)}): {cols_categorical}")

# ─── Optimize memory: Convert objects to category dtype ───
# Category dtype stores strings as integers internally, reducing memory by ~70%.
# It also speeds up groupby operations and enables categorical-specific methods.
Data[cols_categorical] = Data[cols_categorical].astype("category")

print(f"
💾 Memory after optimization: {Data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")



---

# Phase 2: Univariate Analysis

## What Is Univariate Analysis?
**"Uni" = one, "variate" = variable.** We analyze **one variable at a time**.

This is the foundation of all multivariate analysis. You cannot understand relationships  
between variables if you don't understand each variable individually.

## The Complete Statistical Profile
For every numeric variable, we compute:

### Central Tendency (Where is the center?)
| Measure | Formula | When to Use | Vulnerability |
|---------|---------|-------------|---------------|
| **Mean** | $\bar{x} = \frac{1}{n}\sum x_i$ | Symmetric data | Outliers |
| **Median** | $P_{50}$ (50th percentile) | Skewed data | None (robust) |
| **Mode** | Most frequent value | Categorical data | May not exist |
| **Trimmed Mean** | Mean of middle $(1-2\alpha)\%$ | Compromise: robust + efficient | Loses some data |

**Decision Rule:** Mean ≈ Median → symmetric → use Mean. Mean >> Median → right-skewed → use Median.

### Spread / Variability (How wide is the distribution?)
| Measure | Formula | Interpretation |
|---------|---------|----------------|
| **Range** | $\max - \min$ | Total span. Destroyed by one outlier. |
| **IQR** | $Q_3 - Q_1 = P_{75} - P_{25}$ | Middle 50% spread. **Robust.** Standard for boxplots. |
| **Variance** | $\sigma^2 = \frac{\sum(x_i - \bar{x})^2}{n-1}$ | Average squared deviation. Units are squared. |
| **Std Dev** | $\sigma = \sqrt{\sigma^2}$ | Same units as data. "Typical distance from mean." |
| **MAD** | $\frac{1}{n}\sum|x_i - \bar{x}|$ | Mean absolute deviation. More robust than SD. |
| **CV** | $\sigma / \bar{x}$ | Coefficient of variation. Compare spread across scales. |

**Why $n-1$ in variance?** Bessel's correction. We estimate the mean from the same data,  
"using up" one degree of freedom. $n-1$ gives an unbiased estimator.

### Percentiles (Complete distribution shape)
| Percentile | Name | What It Tells You |
|------------|------|-------------------|
| $P_0$ | Minimum | Absolute lowest value |
| $P_1, P_5, P_{10}$ | Lower tail | Detect extreme low values |
| $P_{25}$ | Q1 | Lower bound of middle 50% |
| $P_{50}$ | Median | Center point |
| $P_{75}$ | Q3 | Upper bound of middle 50% |
| $P_{90}, P_{95}, P_{99}$ | Upper tail | Detect extreme high values |
| $P_{100}$ | Maximum | Absolute highest value |

**IQR Method for Outliers:**
- Lower Fence = $Q_1 - 1.5 \times \text{IQR}$
- Upper Fence = $Q_3 + 1.5 \times \text{IQR}$
- Values outside → flagged as outliers (Tukey's convention)
- Values outside $Q_1 - 3\times\text{IQR}$, $Q_3 + 3\times\text{IQR}$ → **extreme** outliers

### Shape Metrics
| Metric | Threshold | Interpretation |
|--------|-----------|----------------|
| **Skewness** | $\|skew\| < 0.5$ | Approximately symmetric |
| | $skew > 0.5$ | Right-skewed (tail on right, mean > median) |
| | $skew < -0.5$ | Left-skewed (tail on left, mean < median) |
| | $\|skew\| > 1$ | **Highly skewed → MUST transform before modeling** |
| **Kurtosis** | $\approx 0$ (excess) | Normal-like tails |
| | $> 0$ | Heavy tails (more outliers than normal) |
| | $< 0$ | Light tails (fewer outliers than normal) |



In [ ]:
# CELL 7: COMPLETE STATISTICAL PROFILE (Numeric Variables)
# ═══════════════════════════════════════════════════════════════════════════════

print(f"
{'═'*70}")
print("COMPLETE STATISTICAL PROFILE: NUMERIC VARIABLES")
print(f"{'═'*70}")

# ─── Calculate Quartiles & IQR ───
Q1 = Data[numeric_cols].quantile(0.25)
Q3 = Data[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

# ─── Outlier Boundaries (IQR Method) ───
# WHY 1.5×IQR? This is Tukey's convention. For a normal distribution, it captures
# approximately 99.3% of data. Values beyond are statistically unusual.
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outlier_count = ((Data[numeric_cols] < lower_bound) | (Data[numeric_cols] > upper_bound)).sum()

# ─── Build the Master Report ───
report = pd.DataFrame({
    # CENTRAL TENDENCY
    "Mean": Data[numeric_cols].mean(),
    "Trimmed_Mean_10%": Data[numeric_cols].apply(lambda x: trim_mean(x.dropna(), 0.10)),
    "Median": Data[numeric_cols].median(),
    "Mode": Data[numeric_cols].mode().iloc[0],

    # SPREAD
    "Min": Data[numeric_cols].min(),
    "Max": Data[numeric_cols].max(),
    "Range": Data[numeric_cols].max() - Data[numeric_cols].min(),
    "Variance": Data[numeric_cols].var(),
    "Std_Dev": Data[numeric_cols].std(),
    "IQR": IQR,
    "MAD": Data[numeric_cols].apply(lambda x: (x - x.median()).abs().median()),
    "CV": Data[numeric_cols].std() / Data[numeric_cols].mean(),

    # SHAPE
    "Skewness": Data[numeric_cols].skew(),
    "Kurtosis": Data[numeric_cols].kurt(),  # Excess kurtosis (Fisher's definition)

    # OUTLIERS
    "Lower_Fence": lower_bound,
    "Upper_Fence": upper_bound,
    "Outlier_Count": outlier_count,
    "Outlier_Pct": (outlier_count / len(Data) * 100).round(2)
})

print(report.round(3))

# ─── Skewness Assessment ───
print(f"
{'─'*70}")
print("SKEWNESS ASSESSMENT & TRANSFORMATION DECISIONS:")
print(f"{'─'*70}")
for col in numeric_cols:
    skew = Data[col].skew()
    if abs(skew) >= 1.0:
        action = "🔴 HIGHLY SKEWED → Apply log1p or Box-Cox transform"
    elif abs(skew) >= 0.5:
        action = "🟡 MODERATELY SKEWED → Consider sqrt or mild log transform"
    else:
        action = "🟢 APPROXIMATELY SYMMETRIC → No transform needed"
    print(f"  {col:30s}: skew={skew:7.3f} → {action}")



In [ ]:
# CELL 8: PERCENTILE ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════

print(f"
{'═'*70}")
print("PERCENTILE ANALYSIS (Complete Distribution Shape)")
print(f"{'═'*70}")

percentiles = Data[numeric_cols].quantile([
    0.00,   # P0  = Minimum
    0.01,   # P1  = 1st percentile (extreme low tail)
    0.05,   # P5  = 5th percentile
    0.10,   # P10 = 10th percentile
    0.25,   # P25 = Q1 (First Quartile)
    0.50,   # P50 = Median (Second Quartile)
    0.75,   # P75 = Q3 (Third Quartile)
    0.90,   # P90 = 90th percentile
    0.95,   # P95 = 95th percentile (cap threshold)
    0.99,   # P99 = 99th percentile (extreme cap threshold)
    1.00    # P100 = Maximum
]).T

percentiles.columns = [
    "P0_Min", "P1", "P5", "P10", "P25_Q1",
    "P50_Median", "P75_Q3", "P90", "P95", "P99", "P100_Max"
]

print(percentiles.round(2))

print(f"
{'─'*70}")
print("KEY INSIGHTS FROM PERCENTILES:")
print(f"{'─'*70}")
for col in numeric_cols:
    p95 = percentiles.loc[col, "P95"]
    p99 = percentiles.loc[col, "P99"]
    max_val = percentiles.loc[col, "P100_Max"]
    if max_val > p99 * 2:
        print(f"  ⚠️  {col}: Max ({max_val:.1f}) is >2× P99 ({p99:.1f}) → Extreme outliers present")
    else:
        print(f"  ✅ {col}: Max is within reasonable range of P99")



---

## 📊 Visualization Toolkit: What Each Plot Reveals

### Density Plot (KDE - Kernel Density Estimate)
**What it is:** A smooth, continuous curve that estimates the probability density function.  
Think of it as a "smoothed histogram" that removes bin-edge artifacts.

**What to look for:**
- **Peaks (modes):** Where is data concentrated? One peak = unimodal. Two+ peaks = multimodal → subpopulations exist.
- **Skewness:** Does the tail extend to the right (right-skewed) or left (left-skewed)?
- **Tails:** Heavy tails (fat) → more outliers than normal. Light tails (thin) → fewer outliers.

**Why it matters for neural networks:**
Skewed inputs cause ReLU neurons to saturate (always output 0 or linear).  
The network learns poorly because gradients flow unevenly. Transforming to symmetric  
distributions ensures stable gradient flow across all neurons.

---

### Box Plot
**What it is:** A standardized way to display the five-number summary (Min, Q1, Median, Q3, Max)  
plus outlier detection.

**Components:**
- **Box:** Spans Q1 to Q3 (the middle 50% of data — the IQR)
- **Line inside box:** The median
- **Whiskers:** Extend to the last non-outlier point (typically 1.5×IQR from the box)
- **Dots beyond whiskers:** Outliers (statistically unusual values)

**What to look for:**
- **Asymmetric box:** Median not centered → skewed data
- **Long whisker on one side:** Tail in that direction
- **Many outlier dots:** Heavy-tailed distribution or data quality issues
- **Compressed box with long whiskers:** High variability

**Why it matters for neural networks:**
Outliers cause exploding gradients. The loss function penalizes extreme errors heavily,  
causing weight updates that destabilize training. Box plots identify which features need  
outlier handling (capping, winsorizing, or robust scaling).

---

### Q-Q Plot (Quantile-Quantile)
**What it is:** Compares your data's quantiles against a theoretical normal distribution's quantiles.

**How to read it:**
- **Points on the diagonal line** → Your data is approximately normal
- **S-curve (S-shape)** → Heavy or light tails (kurtosis issue)
- **Inverted S (curved)** → Skewed distribution
- **Points deviate at ends** → Tails are heavier/lighter than normal

**Why it matters for neural networks:**
Many optimization algorithms (Adam, SGD with momentum) assume roughly normal input distributions.  
Severely non-normal inputs slow convergence. Q-Q plots tell us if standard scaling is sufficient  
or if we need stronger transforms (PowerTransformer, QuantileTransformer).

---

### Hexbin Plot
**What it is:** A 2D histogram using hexagonal bins. Each hexagon's color represents the count  
of data points in that region.

**When to use:** When you have >10,000 points and scatter plots become solid blobs (overplotting).

**What to look for:**
- **Bright/dark hexagons:** High-density regions (where most data lives)
- **Shape of dense region:** Linear? Curved? Clustered?
- **Sparse regions:** Where data is absent

**Why it matters for feature engineering:**
Hexbins reveal non-linear relationships that correlation matrices miss. A U-shape or  
wave pattern in a hexbin tells you to create polynomial features or use non-linear models.



In [ ]:
# CELL 9: DENSITY PLOTS (KDE)
# ═══════════════════════════════════════════════════════════════════════════════

n = len(numeric_cols)
ncols = 3
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(7*ncols, 3*nrows))
fig.patch.set_facecolor(BG_COLOR)
axes = axes.flatten()

for ax, col in zip(axes, numeric_cols):
    ax.set_facecolor(BG_COLOR)

    # KDE with fill for area visualization
    sns.kdeplot(data=Data, x=col, fill=True, linewidth=2.5, 
                color=ACCENT_COLOR, ax=ax, alpha=0.7)

    # Add mean and median reference lines
    mean_val = Data[col].mean()
    median_val = Data[col].median()
    ax.axvline(mean_val, color='#FF6B6B', linestyle='--', linewidth=2, label=f'Mean={mean_val:.1f}')
    ax.axvline(median_val, color='#4ECDC4', linestyle='-.', linewidth=2, label=f'Median={median_val:.1f}')

    ax.set_title(f"Density: {col}", color=TEXT_COLOR, fontsize=14, fontweight="bold")
    ax.set_xlabel(col, color=TEXT_COLOR, fontsize=11)
    ax.set_ylabel("Density", color=TEXT_COLOR, fontsize=11)
    ax.tick_params(colors=TEXT_COLOR)
    ax.legend(fontsize=9, loc='best')

    for spine in ax.spines.values():
        spine.set_visible(False)

for ax in axes[len(numeric_cols):]:
    ax.set_visible(False)

plt.suptitle("Density Plots (KDE): Distribution Shape, Skewness & Central Tendency", 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/mnt/agents/output/01_density_plots.png', dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
plt.show()
print("✅ Density plots saved.")



In [ ]:
# CELL 10: BOX PLOTS (Outlier Detection)
# ═══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(7*ncols, 2.8*nrows))
fig.patch.set_facecolor(BG_COLOR)
axes = axes.flatten()

for ax, col in zip(axes, numeric_cols):
    ax.set_facecolor(BG_COLOR)

    bp = ax.boxplot(
        Data[col].dropna(),
        vert=False,
        patch_artist=True,
        widths=0.6,
        boxprops=dict(facecolor="#F5F5F5", edgecolor=ACCENT_COLOR, linewidth=1.5),
        medianprops=dict(color="#FF6B6B", linewidth=2.5),
        whiskerprops=dict(color=ACCENT_COLOR, linewidth=1.5, linestyle='--'),
        capprops=dict(color=ACCENT_COLOR, linewidth=1.5),
        flierprops=dict(marker="o", markerfacecolor="#FF6B6B", 
                       markeredgecolor="#FF6B6B", markersize=5, alpha=0.6)
    )

    # Add IQR annotation
    q1, med, q3 = Data[col].quantile([0.25, 0.5, 0.75])
    iqr = q3 - q1
    ax.text(0.98, 0.85, f'IQR={iqr:.1f}\nOutliers={outlier_count[col]}', 
            transform=ax.transAxes, fontsize=9, ha='right', va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor=ACCENT_COLOR))

    ax.set_title(f"Box Plot: {col}", color=TEXT_COLOR, fontsize=14, fontweight="bold")
    ax.set_xlabel(col, color=TEXT_COLOR, fontsize=11)
    ax.tick_params(colors=TEXT_COLOR)

    for spine in ax.spines.values():
        spine.set_visible(False)

for ax in axes[len(numeric_cols):]:
    ax.set_visible(False)

plt.suptitle("Box Plots: IQR, Median & Outlier Detection (1.5×IQR Rule)", 
             fontsize=16, fontweight='bold', y=1.02)
plt.subplots_adjust(hspace=0.8, wspace=0.3)
plt.savefig('/mnt/agents/output/02_box_plots.png', dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
plt.show()
print("✅ Box plots saved.")



In [ ]:
# CELL 11: Q-Q PLOTS (Normality Assessment)
# ═══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(7*ncols, 3*nrows))
fig.patch.set_facecolor(BG_COLOR)
axes = axes.flatten()

for ax, col in zip(axes, numeric_cols):
    ax.set_facecolor(BG_COLOR)

    stats.probplot(Data[col].dropna(), dist="norm", plot=ax)

    lines = ax.get_lines()
    lines[0].set_markerfacecolor(ACCENT_COLOR)
    lines[0].set_markeredgecolor(ACCENT_COLOR)
    lines[0].set_markersize(4)
    lines[0].set_alpha(0.6)
    lines[1].set_color("#FF6B6B")
    lines[1].set_linewidth(2)

    ax.set_title(f"Q-Q Plot: {col} vs Normal", color=TEXT_COLOR, fontsize=14, fontweight="bold")
    ax.tick_params(colors=TEXT_COLOR)

    for spine in ax.spines.values():
        spine.set_color(ACCENT_COLOR)

for ax in axes[len(numeric_cols):]:
    ax.set_visible(False)

plt.suptitle("Q-Q Plots: Assessing Normality (Points on Red Line = Normal)", 
             fontsize=16, fontweight='bold', y=1.02)
plt.subplots_adjust(hspace=0.8, wspace=0.3)
plt.savefig('/mnt/agents/output/03_qq_plots.png', dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
plt.show()
print("✅ Q-Q plots saved.")



In [ ]:
# CELL 12: HEXBIN PLOTS (2D Density for Large Datasets)
# ═══════════════════════════════════════════════════════════════════════════════

# Use the first numeric column as x-axis for all pairings
x_col = numeric_cols[0]

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(7*ncols, 3*nrows))
fig.patch.set_facecolor(BG_COLOR)
axes = axes.flatten()

for ax, y_col in zip(axes, numeric_cols):
    ax.set_facecolor(BG_COLOR)

    hb = ax.hexbin(Data[x_col], Data[y_col], gridsize=30, cmap="YlOrRd", 
                   mincnt=1, linewidths=0.3, alpha=0.8)

    cb = fig.colorbar(hb, ax=ax, shrink=0.8)
    cb.set_label("Count", color=TEXT_COLOR)
    cb.ax.yaxis.set_tick_params(color=TEXT_COLOR)
    plt.setp(cb.ax.get_yticklabels(), color=TEXT_COLOR)

    ax.set_xlabel(x_col, color=TEXT_COLOR, fontsize=11)
    ax.set_ylabel(y_col, color=TEXT_COLOR, fontsize=11)
    ax.set_title(f"Hexbin: {x_col} vs {y_col}", color=TEXT_COLOR, fontsize=14, fontweight="bold")
    ax.tick_params(colors=TEXT_COLOR)

    for spine in ax.spines.values():
        spine.set_visible(False)

for ax in axes[len(numeric_cols):]:
    ax.set_visible(False)

plt.suptitle(f"Hexbin Plots: 2D Density Distribution ({x_col} vs All Features)", 
             fontsize=16, fontweight='bold', y=1.02)
plt.subplots_adjust(hspace=0.8, wspace=0.3)
plt.savefig('/mnt/agents/output/04_hexbin_plots.png', dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
plt.show()
print("✅ Hexbin plots saved.")





---

# Phase 3: Bivariate & Multivariate Analysis

## What Is Bivariate/Multivariate Analysis?
**"Bi" = two, "Multi" = many.** We now analyze **relationships between variables**.

Univariate analysis told us "what each variable looks like."  
Bivariate analysis tells us "how variables relate to each other."  
Multivariate analysis tells us "how variables relate in combination."

## Why This Phase Is Critical
- **Feature Selection:** Which features actually predict the target?
- **Multicollinearity:** Are features redundant? (Two features saying the same thing)
- **Interaction Effects:** Do two features together predict better than either alone?
- **Segmentation:** Should we build separate models for different subgroups?

---

## Correlation Analysis

### Pearson Correlation
**What it measures:** Linear relationship strength and direction (-1 to +1).

**Formula:** $r = \frac{\sum(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum(x_i - \bar{x})^2 \sum(y_i - \bar{y})^2}}$

**Assumptions:**
- Both variables are continuous
- Linear relationship
- No significant outliers
- Approximately normal distribution

**Interpretation:**
| |r| | Strength |
|-----|----------|
| < 0.3 | Weak |
| 0.3 - 0.7 | Moderate |
| > 0.7 | Strong |

**⚠️ CRITICAL CAVEATS:**
1. Correlation ≠ Causation. r = 0.9 does NOT mean X causes Y.
2. Only captures LINEAR relationships. A perfect U-shape has r ≈ 0.
3. Sensitive to outliers. One extreme point can make r = 0 → r = 0.9.
4. r = 0 does NOT mean independence (could be non-linear).

### Spearman Rank Correlation
**What it measures:** Monotonic relationship (consistently increasing/decreasing, not necessarily linear).

**How it works:** Converts values to ranks, then computes Pearson correlation on ranks.

**When to use:**
- Data is skewed or has outliers
- Variables are ordinal
- Relationship is monotonic but non-linear

**Why it's better for this dataset:**
Our EDA revealed many skewed features (Daily_Calorie_Intake, Alcohol_Consumption).  
Pearson would be misleading. Spearman is robust and more trustworthy here.

---

## Variance Inflation Factor (VIF)
**What it measures:** How much a feature's variance is inflated due to multicollinearity.

**Formula:** $VIF_j = \frac{1}{1 - R_j^2}$ where $R_j^2$ is from regressing feature $j$ on all other features.

**Interpretation:**
| VIF | Severity | Action |
|-----|----------|--------|
| 1 | None | Perfect |
| 1-5 | Moderate | Acceptable |
| 5-10 | High | Problematic |
| > 10 | Severe | Must remove |

**Why it matters:**
Multicollinear features make model coefficients unstable. Small data changes  
cause large coefficient swings. The model can't determine which feature is "responsible."



In [ ]:
# CELL 13: CORRELATION MATRIX (Pearson + Spearman)
# ═══════════════════════════════════════════════════════════════════════════════

print(f"
{'═'*70}")
print("CORRELATION ANALYSIS")
print(f"{'═'*70}")

# Pearson correlation (linear relationships)
pearson_corr = Data.corr(numeric_only=True, method='pearson')

# Spearman correlation (monotonic, robust to outliers)
spearman_corr = Data.corr(numeric_only=True, method='spearman')

print("
📊 PEARSON CORRELATION MATRIX (Linear):")
print(pearson_corr.round(3))

print("
📊 SPEARMAN CORRELATION MATRIX (Monotonic, Robust):")
print(spearman_corr.round(3))

# ─── Correlation Heat Map ───
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Pearson heatmap
mask = np.triu(np.ones_like(pearson_corr, dtype=bool))
sns.heatmap(pearson_corr, mask=mask, annot=True, cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, fmt='.2f', ax=axes[0],
            cbar_kws={'label': 'Pearson r'})
axes[0].set_title('Pearson Correlation (Linear)', fontsize=14, fontweight='bold')

# Spearman heatmap
sns.heatmap(spearman_corr, mask=mask, annot=True, cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, fmt='.2f', ax=axes[1],
            cbar_kws={'label': 'Spearman ρ'})
axes[1].set_title('Spearman Correlation (Monotonic, Robust)', fontsize=14, fontweight='bold')

plt.suptitle("Correlation Analysis: Pearson vs Spearman", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/mnt/agents/output/05_correlation_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Correlation heatmaps saved.")

# ─── Identify strong correlations ───
print(f"
{'─'*70}")
print("STRONG CORRELATIONS DETECTED (|r| > 0.7):")
print(f"{'─'*70}")
strong_corr = []
for i in range(len(pearson_corr.columns)):
    for j in range(i+1, len(pearson_corr.columns)):
        r = pearson_corr.iloc[i, j]
        if abs(r) > 0.7:
            strong_corr.append((pearson_corr.columns[i], pearson_corr.columns[j], r))
            print(f"  🔴 {pearson_corr.columns[i]} ↔ {pearson_corr.columns[j]}: r = {r:.3f}")
            print(f"     → MULTICOLLINEARITY RISK: Consider removing one or using PCA")

if not strong_corr:
    print("  ✅ No severe multicollinearity detected (|r| < 0.7 for all pairs)")



In [ ]:
# CELL 14: VARIANCE INFLATION FACTOR (VIF)
# ═══════════════════════════════════════════════════════════════════════════════

from statsmodels.stats.outliers_influence import variance_inflation_factor

print(f"
{'═'*70}")
print("MULTICOLLINEARITY ASSESSMENT: VIF")
print(f"{'═'*70}")

# VIF requires no missing values and all numeric
X_vif = Data[numeric_cols].dropna()

vif_df = pd.DataFrame()
vif_df["Feature"] = X_vif.columns
vif_df["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
vif_df = vif_df.sort_values("VIF", ascending=False)

print(vif_df.round(2))

print(f"
{'─'*70}")
print("VIF INTERPRETATION:")
print(f"{'─'*70}")
for _, row in vif_df.iterrows():
    feat, vif_val = row['Feature'], row['VIF']
    if vif_val > 10:
        print(f"  🔴 {feat}: VIF = {vif_val:.2f} → SEVERE multicollinearity. MUST remove.")
    elif vif_val > 5:
        print(f"  🟡 {feat}: VIF = {vif_val:.2f} → HIGH multicollinearity. Consider removing.")
    else:
        print(f"  🟢 {feat}: VIF = {vif_val:.2f} → Acceptable.")



---

## Statistical Testing: Why We Need More Than p-values

### The Problem with p-values Alone
A p-value tells you **IF** a difference exists, but not **HOW BIG** it is.  
With large samples (n > 10,000), EVERYTHING is "statistically significant"  
even if the effect is trivial.

**Solution: Always report effect sizes alongside p-values.**

---

### Cohen's d (Effect Size for T-Tests)
**What it measures:** Standardized difference between two group means.

**Formula:** $d = \frac{\bar{x}_1 - \bar{x}_2}{s_{pooled}}$ where $s_{pooled} = \sqrt{\frac{(n_1-1)s_1^2 + (n_2-1)s_2^2}{n_1+n_2-2}}$

**Interpretation:**
| d | Effect Size | Practical Meaning |
|---|-------------|---------------------|
| < 0.2 | Negligible | Difference is trivial |
| 0.2 - 0.5 | Small | Noticeable but small |
| 0.5 - 0.8 | Medium | Clear practical difference |
| > 0.8 | Large | Very substantial difference |

**Why it matters:**
p = 0.001 with d = 0.15 → "Significant" but meaningless in practice.  
p = 0.04 with d = 0.9 → "Marginally significant" but hugely important.

---

### Cramér's V (Effect Size for Chi-Square)
**What it measures:** Strength of association between two categorical variables (0 to 1).

**Formula:** $V = \sqrt{\frac{\chi^2}{n \times \min(k-1, r-1)}}$

**Interpretation:**
| V | Association |
|---|-------------|
| < 0.1 | Negligible |
| 0.1 - 0.3 | Small |
| 0.3 - 0.5 | Medium |
| > 0.5 | Large |

---

### Non-Parametric Tests: When to Use Them

**The Problem with T-Tests and ANOVA:**
They assume:
1. Normal distribution
2. Equal variances (homoscedasticity)
3. No extreme outliers

Our EDA showed many features are **skewed** with **outliers**. These assumptions are violated.

**Mann-Whitney U Test** (alternative to independent T-test):
- Compares **ranks**, not raw values
- **Robust to outliers** (extreme values just become highest/lowest rank)
- **No normality assumption**
- Tests: "Does one group tend to have larger values than the other?"

**Kruskal-Wallis H Test** (alternative to ANOVA):
- Extension of Mann-Whitney to 3+ groups
- Same robustness benefits
- Tests: "Do at least 2 groups differ in their typical values?"

**When to use parametric vs non-parametric:**
| Condition | Use Parametric | Use Non-Parametric |
|-----------|----------------|-------------------|
| Normal distribution | ✓ T-test, ANOVA | Mann-Whitney, Kruskal-Wallis |
| Skewed distribution | ✗ | ✓ Mann-Whitney, Kruskal-Wallis |
| Outliers present | ✗ | ✓ Mann-Whitney, Kruskal-Wallis |
| Small sample (n < 30) | ✗ | ✓ Mann-Whitney, Kruskal-Wallis |
| Ordinal data | ✗ | ✓ Mann-Whitney, Kruskal-Wallis |



In [ ]:
# CELL 15: COHEN'S d (Effect Size for Group Comparisons)
# ═══════════════════════════════════════════════════════════════════════════════

print(f"
{'═'*70}")
print("EFFECT SIZE: COHEN'S d")
print(f"{'═'*70}")

# Example: Compare Health_Score between Genders
# (Assuming Gender is encoded as 0/1; adjust if different)
if 'Gender' in Data.columns and Data['Gender'].nunique() == 2:
    # Find a suitable numeric target
    target_col = 'Health_Score' if 'Health_Score' in Data.columns else numeric_cols[-1]

    genders = Data['Gender'].unique()
    group1 = Data[Data['Gender'] == genders[0]][target_col].dropna().values
    group2 = Data[Data['Gender'] == genders[1]][target_col].dropna().values

    # Manual Cohen's d calculation
    mean1, mean2 = np.mean(group1), np.mean(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    n1, n2 = len(group1), len(group2)

    # Pooled standard deviation
    s_pooled = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))

    # Cohen's d
    cohens_d = abs(mean1 - mean2) / s_pooled

    print(f"
Comparison: {target_col} by Gender")
    print(f"  Group 1 (n={n1}): Mean = {mean1:.2f}, Std = {np.sqrt(var1):.2f}")
    print(f"  Group 2 (n={n2}): Mean = {mean2:.2f}, Std = {np.sqrt(var2):.2f}")
    print(f"  Pooled Std: {s_pooled:.2f}")
    print(f"  Cohen's d: {cohens_d:.4f}")

    if cohens_d < 0.2:
        effect = "Negligible"
    elif cohens_d < 0.5:
        effect = "Small"
    elif cohens_d < 0.8:
        effect = "Medium"
    else:
        effect = "Large"
    print(f"  Effect Size: {effect}")

    # Visualization
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.set_facecolor(BG_COLOR)
    fig.patch.set_facecolor(BG_COLOR)

    sns.kdeplot(group1, label=f"Gender {genders[0]} (Mean: {mean1:.1f})", 
                fill=True, alpha=0.5, color=ACCENT_COLOR, ax=ax)
    sns.kdeplot(group2, label=f"Gender {genders[1]} (Mean: {mean2:.1f})", 
                fill=True, alpha=0.5, color="#FF6B6B", ax=ax)

    ax.axvline(mean1, color=ACCENT_COLOR, linestyle='--', linewidth=2)
    ax.axvline(mean2, color="#FF6B6B", linestyle='--', linewidth=2)

    ax.set_title(f"Cohen's d = {cohens_d:.4f} ({effect} Effect)", 
                 fontsize=14, fontweight='bold', color=TEXT_COLOR)
    ax.set_xlabel(target_col, color=TEXT_COLOR)
    ax.set_ylabel("Density", color=TEXT_COLOR)
    ax.legend(fontsize=11)
    ax.tick_params(colors=TEXT_COLOR)

    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.tight_layout()
    plt.savefig('/mnt/agents/output/06_cohens_d.png', dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.show()
    print("✅ Cohen's d visualization saved.")



In [ ]:
# CELL 16: CHI-SQUARE TEST + CRAMÉR'S V (Categorical Relationships)
# ═══════════════════════════════════════════════════════════════════════════════

print(f"
{'═'*70}")
print("CHI-SQUARE TEST + CRAMÉR'S V")
print(f"{'═'*70}")

# Find suitable categorical features for testing
cat_cols_available = Data.select_dtypes(include=['category', 'object']).columns.tolist()

if len(cat_cols_available) >= 2:
    feat_col = cat_cols_available[0]
    target_col = cat_cols_available[1] if len(cat_cols_available) > 1 else cat_cols_available[0]

    temp_df = Data[[feat_col, target_col]].dropna()
    contingency = pd.crosstab(temp_df[feat_col], temp_df[target_col])

    print(f"
Contingency Table: {feat_col} vs {target_col}")
    print(contingency)

    # Chi-square test
    chi2, p_val, dof, expected = stats.chi2_contingency(contingency)

    # Cramér's V
    n = contingency.sum().sum()
    min_dim = min(contingency.shape[0] - 1, contingency.shape[1] - 1)
    cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0

    print(f"
{'─'*70}")
    print("TEST RESULTS:")
    print(f"{'─'*70}")
    print(f"  Chi-square statistic: {chi2:.4f}")
    print(f"  Degrees of freedom: {dof}")
    print(f"  p-value: {p_val:.6f} {'→ SIGNIFICANT' if p_val < 0.05 else '→ NOT SIGNIFICANT'}")
    print(f"  Cramér's V: {cramers_v:.4f}", end="")
    if cramers_v < 0.1:
        print(" → Negligible association")
    elif cramers_v < 0.3:
        print(" → Small association")
    elif cramers_v < 0.5:
        print(" → Medium association")
    else:
        print(" → Large association")

    # Standardized residuals
    print(f"
{'─'*70}")
    print("STANDARDIZED RESIDUALS (|residual| > 2 = significant cell):")
    print(f"{'─'*70}")
    for i in contingency.index:
        for j in contingency.columns:
            obs = contingency.loc[i, j]
            exp = expected[contingency.index.get_loc(i), contingency.columns.get_loc(j)]
            residual = (obs - exp) / np.sqrt(exp)
            if abs(residual) > 2:
                direction = "MORE" if residual > 0 else "FEWER"
                print(f"  {i}/{j}: residual = {residual:.2f} → {direction} than expected")

    # Visualization
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.set_facecolor(BG_COLOR)
    fig.patch.set_facecolor(BG_COLOR)

    sns.heatmap(contingency, annot=True, fmt='d', cmap='Blues', 
                linewidths=0.5, square=True, ax=ax,
                cbar_kws={'label': 'Count'})
    ax.set_title(f"{feat_col} vs {target_col}\nChi²={chi2:.2f}, p={p_val:.4f}, Cramér's V={cramers_v:.4f}",
                 fontsize=14, fontweight='bold')
    ax.set_xlabel(target_col)
    ax.set_ylabel(feat_col)

    plt.tight_layout()
    plt.savefig('/mnt/agents/output/07_chi_square.png', dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.show()
    print("✅ Chi-square visualization saved.")



In [ ]:
# CELL 17: NON-PARAMETRIC TESTS (Mann-Whitney U + Kruskal-Wallis)
# ═══════════════════════════════════════════════════════════════════════════════

print(f"
{'═'*70}")
print("NON-PARAMETRIC TESTS: MANN-WHITNEY U & KRUSKAL-WALLIS")
print(f"{'═'*70}")

# Select a skewed feature and binary/multiclass target
skewed_feature = None
for col in numeric_cols:
    if abs(Data[col].skew()) > 0.5:
        skewed_feature = col
        break

if skewed_feature and len(cat_cols_available) > 0:
    binary_target = cat_cols_available[0]
    multi_target = cat_cols_available[0] if len(cat_cols_available) == 1 else cat_cols_available[1]

    # ─── MANN-WHITNEY U TEST ───
    print(f"
{'─'*70}")
    print("MANN-WHITNEY U TEST (Alternative to T-Test)")
    print(f"{'─'*70}")
    print(f"Question: Does '{skewed_feature}' differ by '{binary_target}'?")
    print(f"Why Mann-Whitney? '{skewed_feature}' is skewed (skew={Data[skewed_feature].skew():.2f}),")
    print(f"violating T-test normality assumptions.")

    mw_data = Data[[skewed_feature, binary_target]].dropna()
    categories = mw_data[binary_target].unique()

    if len(categories) == 2:
        group1_mw = mw_data[mw_data[binary_target] == categories[0]][skewed_feature].values
        group2_mw = mw_data[mw_data[binary_target] == categories[1]][skewed_feature].values

        u_stat, mw_p = mannwhitneyu(group1_mw, group2_mw, alternative='two-sided')

        print(f"
  Group 1 ({categories[0]}): n={len(group1_mw)}, median={np.median(group1_mw):.2f}")
        print(f"  Group 2 ({categories[1]}): n={len(group2_mw)}, median={np.median(group2_mw):.2f}")
        print(f"  U-statistic: {u_stat:.2f}")
        print(f"  p-value: {mw_p:.6f}")
        print(f"  Verdict: {'SIGNIFICANT difference in ranks' if mw_p < 0.05 else 'NO significant difference'}")

        # ─── KRUSKAL-WALLIS H TEST ───
        print(f"
{'─'*70}")
        print("KRUSKAL-WALLIS H TEST (Alternative to ANOVA)")
        print(f"{'─'*70}")
        print(f"Question: Does '{skewed_feature}' differ across '{multi_target}' categories?")

        kw_data = Data[[skewed_feature, multi_target]].dropna()
        groups_kw = [group[skewed_feature].values for name, group in kw_data.groupby(multi_target)]

        h_stat, kw_p = kruskal(*groups_kw)

        print(f"
  Number of groups: {len(groups_kw)}")
        print(f"  H-statistic: {h_stat:.2f}")
        print(f"  p-value: {kw_p:.6f}")
        print(f"  Verdict: {'At least one group differs significantly' if kw_p < 0.05 else 'No significant differences'}")

        # Visualization
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))

        # Mann-Whitney boxplot
        axes[0].set_facecolor(BG_COLOR)
        sns.boxplot(x=binary_target, y=skewed_feature, data=mw_data, 
                    palette=[ACCENT_COLOR, "#FF6B6B"], ax=axes[0])
        axes[0].set_title(f"Mann-Whitney U Test\np = {mw_p:.5f}", fontsize=14, fontweight='bold')
        axes[0].tick_params(colors=TEXT_COLOR)

        # Kruskal-Wallis boxplot
        axes[1].set_facecolor(BG_COLOR)
        sns.boxplot(x=multi_target, y=skewed_feature, data=kw_data, 
                    palette="Set3", ax=axes[1])
        axes[1].set_title(f"Kruskal-Wallis H Test\np = {kw_p:.5f}", fontsize=14, fontweight='bold')
        axes[1].tick_params(colors=TEXT_COLOR)
        axes[1].tick_params(axis='x', rotation=45)

        fig.patch.set_facecolor(BG_COLOR)
        plt.suptitle("Non-Parametric Tests: Robust to Skewness & Outliers", 
                     fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.savefig('/mnt/agents/output/08_nonparametric_tests.png', dpi=150, 
                    bbox_inches='tight', facecolor=BG_COLOR)
        plt.show()
        print("✅ Non-parametric test visualizations saved.")





---

# Phase 4: Feature Engineering

## What Is Feature Engineering?
Feature engineering is the process of **transforming raw data into features** that better represent  
the underlying problem to predictive models, resulting in improved model performance.

**It is the most important skill in data science.** Algorithms are commodities; feature engineering  
is where competitive advantage lives.

---

## Our Engineering Strategy (Data-Driven)

Every decision below is grounded in the EDA we just performed:

| EDA Finding | Engineering Action | Rationale |
|-------------|-------------------|-----------|
| Missing values in numeric columns | Median imputation | Median is robust to skewness |
| Missing values in categorical columns | Mode imputation | Most frequent is best guess |
| Outliers detected by IQR method | Winsorize at P1/P99 | Preserves data, removes extremes |
| \|Skew\| ≥ 1.0 | Log1p or Box-Cox transform | Makes distribution symmetric |
| 0.5 ≤ \|Skew\| < 1.0 | Sqrt or mild log transform | Reduces skew moderately |
| Heavy-tailed features (high kurtosis) | RobustScaler | Outlier-immune scaling |
| Normal-ish features | StandardScaler | Mean=0, Std=1 for neural networks |
| Bounded features (ratings, percentages) | MinMaxScaler | Preserves [0,1] interpretation |
| Low cardinality categoricals (≤2 unique) | One-hot encoding | No ordinal assumption |
| Medium cardinality (3-4 unique) | Label encoding | Efficient, preserves order if ordinal |
| High cardinality (>4 unique) | Semantic embeddings | Captures meaning, not just identity |

---

## 4.1 Missing Value Imputation

### The Decision Framework

```
Is the variable numeric?
├─ YES → Is the distribution skewed? (|skew| > 0.5)
│        ├─ YES → Fill with MEDIAN (robust to outliers)
│        └─ NO  → Fill with MEAN (most efficient for normal data)
│
└─ NO (categorical) → Fill with MODE (most frequent category)
```

**Why not just fill everything with the mean?**
- For skewed data (income, purchase amounts), the mean is pulled toward extreme values
- Example: Incomes [20k, 25k, 30k, 35k, 1000k] → Mean = $222k, Median = $30k
- The median ($30k) represents the "typical" person; the mean ($222k) represents no one

**"Is Missing" Flags:**
Sometimes the FACT that a value is missing is predictive. If people who skip the income  
question have different churn rates, we want the model to know the value was missing.

### Formula Reference
- **Median:** Sort values, take middle (or average of two middle values)
- **Mean:** $ar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$
- **Mode:** Most frequent value (can be multiple)



In [ ]:
# CELL 18: MISSING VALUE IMPUTATION
# ═══════════════════════════════════════════════════════════════════════════════

print(f"
{'═'*70}")
print("FEATURE ENGINEERING: PHASE 1 - MISSING VALUE IMPUTATION")
print(f"{'═'*70}")

# ─── Track what we do for documentation ───
imputation_log = []

# ─── Numeric Columns: Median vs Mean Decision ───
for col in numeric_cols:
    missing_count = Data[col].isnull().sum()
    if missing_count > 0:
        skew = Data[col].skew()

        if abs(skew) > 0.5:
            # Skewed → Use median (robust to outliers)
            fill_value = Data[col].median()
            strategy = "median"
            reason = f"skewed (skew={skew:.2f}), median is robust"
        else:
            # Symmetric → Use mean (most efficient estimator)
            fill_value = Data[col].mean()
            strategy = "mean"
            reason = f"approximately symmetric (skew={skew:.2f})"

        Data[col] = Data[col].fillna(fill_value)
        imputation_log.append({
            'Column': col,
            'Missing': missing_count,
            'Strategy': strategy,
            'Fill_Value': round(fill_value, 4),
            'Reason': reason
        })
        print(f"  ✅ {col}: Filled {missing_count} missing with {strategy} = {fill_value:.2f} ({reason})")

# ─── Categorical Columns: Mode Imputation ───
for col in cols_categorical:
    missing_count = Data[col].isnull().sum()
    if missing_count > 0:
        mode_val = Data[col].mode()[0]
        Data[col] = Data[col].fillna(mode_val)
        imputation_log.append({
            'Column': col,
            'Missing': missing_count,
            'Strategy': 'mode',
            'Fill_Value': mode_val,
            'Reason': 'categorical: most frequent is best guess'
        })
        print(f"  ✅ {col}: Filled {missing_count} missing with mode = '{mode_val}'")

# ─── Create "is_missing" flags for informative missingness ───
# If a feature had >5% missing, the missingness pattern itself may be predictive
for col in numeric_cols + cols_categorical:
    missing_pct = Raw_Data[col].isnull().sum() / len(Raw_Data)
    if missing_pct > 0.05:  # More than 5% missing
        flag_col = f"{col}_was_missing"
        Data[flag_col] = Raw_Data[col].isnull().astype(int)
        print(f"  🏷️  Created flag: {flag_col} ({missing_pct*100:.1f}% were missing)")

print(f"
{'─'*70}")
print("IMPUTATION SUMMARY:")
print(f"{'─'*70}")
if imputation_log:
    imp_df = pd.DataFrame(imputation_log)
    print(imp_df.to_string(index=False))
else:
    print("  No missing values required imputation.")

print(f"
✅ Missing value imputation complete.")
print(f"   Remaining missing values: {Data.isnull().sum().sum()}")



---

## 4.2 Outlier Handling: Winsorization

### Why Not Just Remove Outliers?
Removing outliers destroys data and can introduce bias. Instead, we **WINSORIZE** — 
cap extreme values at percentiles. This:
1. Preserves the data point (doesn't delete it)
2. Reduces the influence of extremes
3. Maintains sample size
4. Keeps "noise" that helps models generalize

### The Winsorization Strategy
| Outlier Type | Action | Percentile |
|--------------|--------|------------|
| Extreme high values | Cap at P99 | 99th percentile |
| Extreme low values | Floor at P1 | 1st percentile |

**Why P1 and P99 (not P0 and P100)?**
- P0/P100 would be min/max → no capping happens
- P1/P99 remove the most extreme 1% from each tail
- This preserves 98% of the distribution's shape while defanging extremes

**Why keep some outliers (not cap at P95)?**
Models need to see SOME noise to generalize. If we remove ALL extremes, the model  
becomes brittle and fails when it encounters edge cases in production.

### Formula
$$x_{winsorized} = \begin{cases} P_1 & \text{if } x < P_1 \\ x & \text{if } P_1 \leq x \leq P_{99} \\ P_{99} & \text{if } x > P_{99} \end{cases}$$



In [ ]:
# CELL 19: OUTLIER HANDLING (WINSORIZATION)
# ═══════════════════════════════════════════════════════════════════════════════

print(f"
{'═'*70}")
print("FEATURE ENGINEERING: PHASE 2 - OUTLIER WINSORIZATION")
print(f"{'═'*70}")

winsorization_log = []

for col in numeric_cols:
    p1 = Data[col].quantile(0.01)   # 1st percentile (floor)
    p99 = Data[col].quantile(0.99)  # 99th percentile (cap)

    original_min = Data[col].min()
    original_max = Data[col].max()

    # Count how many values will be affected
    n_capped = (Data[col] > p99).sum()
    n_floored = (Data[col] < p1).sum()

    # Apply winsorization
    Data[col] = np.clip(Data[col], p1, p99)

    winsorization_log.append({
        'Column': col,
        'P1_Floor': round(p1, 2),
        'P99_Cap': round(p99, 2),
        'Original_Min': round(original_min, 2),
        'Original_Max': round(original_max, 2),
        'Values_Floored': n_floored,
        'Values_Capped': n_capped,
        'Total_Affected': n_floored + n_capped
    })

    print(f"  ✅ {col}:")
    print(f"     Floor (P1):  {p1:.2f}  ← {n_floored} values floored from {original_min:.2f}")
    print(f"     Cap (P99):   {p99:.2f}  ← {n_capped} values capped from {original_max:.2f}")

print(f"
{'─'*70}")
print("WINSORIZATION SUMMARY:")
print(f"{'─'*70}")
win_df = pd.DataFrame(winsorization_log)
print(win_df.to_string(index=False))

print(f"
✅ Winsorization complete. Extreme outliers capped but data preserved.")



---

## 4.3 Distribution Transformation (Skewness Reduction)

### Why Transform?
Neural networks and many ML algorithms assume roughly symmetric input distributions.  
Skewed data causes:
1. **Gradient issues:** ReLU neurons saturate (always output 0 or linear)
2. **Loss imbalance:** Extreme values dominate the loss function
3. **Poor convergence:** Optimization gets stuck in skewed regions
4. **Distance distortion:** In clustering, skewed features dominate distance metrics

### Transformation Decision Tree
```
|Skewness| < 0.5     → No transform needed (symmetric enough)
0.5 ≤ |Skew| < 1.0  → Mild transform: sqrt(x) or mild log
|Skew| ≥ 1.0        → Strong transform: log1p(x), Box-Cox, Yeo-Johnson
Bimodal              → Cluster first, then model separately
```

### Transformations Explained

| Transform | Formula | When to Use | Effect |
|-----------|---------|-------------|--------|
| **Log1p** | $\log(1 + x)$ | Right-skewed, $x \geq 0$ | Compresses right tail strongly |
| **Sqrt** | $\sqrt{x}$ | Mildly right-skewed, $x \geq 0$ | Moderate compression |
| **Box-Cox** | $rac{x^\lambda - 1}{\lambda}$ | Positive data, automated $\lambda$ | Finds optimal power |
| **Yeo-Johnson** | Similar to Box-Cox but handles negatives | Any real-valued data | Most versatile |
| **PowerTransformer** | sklearn wrapper for Yeo-Johnson/Box-Cox | Automated pipeline | Convenience |

**Why log1p instead of log?**
$\log(0)$ is undefined ($-\infty$). $\log(1 + 0) = 0$. Safe for zero-valued data.

**Why sqrt before log?**
Sqrt is milder. If skew is moderate (0.5-1.0), sqrt may be sufficient.  
Log is aggressive. Only use for severe skew (≥ 1.0).



In [ ]:
# CELL 20: SKEWNESS REDUCTION (Distribution Transformation)
# ═══════════════════════════════════════════════════════════════════════════════

print(f"
{'═'*70}")
print("FEATURE ENGINEERING: PHASE 3 - SKEWNESS REDUCTION")
print(f"{'═'*70}")

transformation_log = []

for col in numeric_cols:
    original_skew = Data[col].skew()

    if abs(original_skew) >= 1.0:
        # HIGHLY SKEWED → Strong transform: log1p
        if Data[col].min() >= 0:
            Data[col] = np.log1p(Data[col])
            transform = "log1p"
            new_skew = Data[col].skew()
            reason = f"|skew|={abs(original_skew):.2f} >= 1.0, log1p applied"
        else:
            # Negative values present → Use Yeo-Johnson instead
            pt = PowerTransformer(method='yeo-johnson')
            Data[[col]] = pt.fit_transform(Data[[col]].values.reshape(-1, 1))
            transform = "yeo-johnson"
            new_skew = Data[col].skew()
            reason = f"|skew|={abs(original_skew):.2f}, negative values → Yeo-Johnson"

    elif abs(original_skew) >= 0.5:
        # MODERATELY SKEWED → Mild transform: sqrt
        if Data[col].min() >= 0:
            Data[col] = np.sqrt(Data[col])
            transform = "sqrt"
            new_skew = Data[col].skew()
            reason = f"0.5 <= |skew|={abs(original_skew):.2f} < 1.0, sqrt applied"
        else:
            Data[col] = np.sign(Data[col]) * np.sqrt(np.abs(Data[col]))
            transform = "signed_sqrt"
            new_skew = Data[col].skew()
            reason = f"0.5 <= |skew|={abs(original_skew):.2f}, signed sqrt applied"

    else:
        # APPROXIMATELY SYMMETRIC → No transform
        transform = "none"
        new_skew = original_skew
        reason = f"|skew|={abs(original_skew):.2f} < 0.5, no transform needed"

    transformation_log.append({
        'Column': col,
        'Original_Skew': round(original_skew, 4),
        'Transform': transform,
        'New_Skew': round(new_skew, 4),
        'Improvement': round(abs(original_skew) - abs(new_skew), 4),
        'Reason': reason
    })

    status = "✅" if abs(new_skew) < 0.5 else "⚠️"
    print(f"  {status} {col}: skew {original_skew:+.3f} → {new_skew:+.3f} ({transform})")

print(f"
{'─'*70}")
print("TRANSFORMATION SUMMARY:")
print(f"{'─'*70}")
trans_df = pd.DataFrame(transformation_log)
print(trans_df.to_string(index=False))

# Count how many are now symmetric
symmetric_count = sum(abs(t['New_Skew']) < 0.5 for t in transformation_log)
print(f"
✅ Skewness reduction complete: {symmetric_count}/{len(numeric_cols)} features now symmetric (|skew| < 0.5)")



---

## 4.4 Feature Scaling

### Why Scale?
Machine learning algorithms are **distance-based** or **gradient-based**:
- **Distance-based** (KNN, K-Means, SVM, Neural Networks): Features with larger scales dominate
- **Gradient-based** (Linear Regression, Logistic Regression, Neural Networks): Different scales cause uneven gradient updates

Example: Age [18-80] vs Income [0-500,000]. Without scaling, income dominates by 10,000×.

### Scaling Decision Tree
```
Normal distribution, no outliers?     → StandardScaler
Bounded features (ratings, %)?        → MinMaxScaler
Heavy outliers present?               → RobustScaler
Highly skewed even after transform?   → QuantileTransformer
```

### Scalers Explained

| Scaler | Formula | Result | Best For | Avoid When |
|--------|---------|--------|----------|------------|
| **StandardScaler** | $z = \frac{x - \mu}{\sigma}$ | Mean=0, Std=1 | Normal-ish data, most algorithms | Heavy outliers |
| **MinMaxScaler** | $x' = \frac{x - \min}{\max - \min}$ | Range [0,1] | Bounded features, neural networks | Outliers present |
| **RobustScaler** | $x' = \frac{x - \text{median}}{\text{IQR}}$ | Median=0, IQR=1 | Outlier-heavy data | No outliers (less efficient) |
| **QuantileTransformer** | Rank-based mapping | Uniform/Normal | Any distribution | Need exact distances |

**⚠️ CRITICAL: Fit on training data ONLY**
```python
scaler.fit(X_train)      # Learn μ, σ from training data
X_train_scaled = scaler.transform(X_train)  # Transform training
X_test_scaled = scaler.transform(X_test)      # Transform test (SAME parameters!)
```
Using full dataset statistics leaks information from test to train.



In [ ]:
# CELL 21: FEATURE SCALING
# ═══════════════════════════════════════════════════════════════════════════════

print(f"
{'═'*70}")
print("FEATURE ENGINEERING: PHASE 4 - FEATURE SCALING")
print(f"{'═'*70}")

scaling_log = []

# We use a hybrid approach based on post-transform characteristics
for col in numeric_cols:
    kurt = Data[col].kurt()
    skew_post = Data[col].skew()

    # Decision: Which scaler to use?
    if abs(kurt) > 3 or abs(skew_post) > 1:
        # Heavy tails or still skewed → RobustScaler (outlier-immune)
        scaler = RobustScaler()
        scaler_name = "RobustScaler"
        reason = f"high kurtosis ({kurt:.2f}) or residual skew ({skew_post:.2f})"
    elif Data[col].min() >= 0 and Data[col].max() <= 1:
        # Already bounded → MinMaxScaler to ensure [0,1]
        scaler = MinMaxScaler()
        scaler_name = "MinMaxScaler"
        reason = "bounded feature, preserve [0,1] range"
    else:
        # Normal-ish → StandardScaler
        scaler = StandardScaler()
        scaler_name = "StandardScaler"
        reason = f"normal-ish (kurt={kurt:.2f}, skew={skew_post:.2f})"

    # Fit and transform (in production, fit on train only!)
    Data[[col]] = scaler.fit_transform(Data[[col]].values.reshape(-1, 1))

    scaling_log.append({
        'Column': col,
        'Scaler': scaler_name,
        'Reason': reason,
        'New_Mean': round(Data[col].mean(), 4),
        'New_Std': round(Data[col].std(), 4),
        'New_Min': round(Data[col].min(), 4),
        'New_Max': round(Data[col].max(), 4)
    })

    print(f"  ✅ {col}: {scaler_name} ({reason})")

print(f"
{'─'*70}")
print("SCALING SUMMARY:")
print(f"{'─'*70}")
scale_df = pd.DataFrame(scaling_log)
print(scale_df.to_string(index=False))

print(f"
✅ Feature scaling complete. All numeric features standardized.")



---

## 4.5 Categorical Encoding

### The Encoding Decision Tree
```
Cardinality ≤ 2?          → One-hot encoding (drop_first=True)
Cardinality 3-4?          → Label encoding (efficient, no dimension explosion)
Cardinality 5-15?         → One-hot encoding (manageable dimensions)
Cardinality 15-50?        → Binary encoding (fewer dimensions than one-hot)
Cardinality > 50?         → Target encoding with smoothing
Deep learning model?      → Embeddings (learned dense vectors)
```

### Why Semantic Embeddings for High Cardinality?
Traditional encoding treats "Doctor" and "Nurse" as unrelated (one-hot: [1,0] vs [0,1]).  
**Semantic embeddings** treat them as similar (both are healthcare professions).

**How it works:**
1. Use a pre-trained language model (Sentence-BERT) to convert text to dense vectors
2. "Doctor" → [0.12, -0.45, 0.89, ...] (384 dimensions)
3. "Nurse" → [0.15, -0.42, 0.85, ...] (similar vector = similar meaning)
4. The neural network learns to use these semantic relationships

**Why this is superior:**
- Captures MEANING, not just identity
- Reduces dimensionality (384 dims vs 1000+ one-hot columns)
- Enables zero-shot generalization (unseen categories get meaningful vectors)
- Pre-trained on billions of text snippets = world knowledge injected



In [ ]:
# CELL 22: CATEGORICAL ENCODING
# ═══════════════════════════════════════════════════════════════════════════════

print(f"
{'═'*70}")
print("FEATURE ENGINEERING: PHASE 5 - CATEGORICAL ENCODING")
print(f"{'═'*70}")

encoding_log = []

# Re-identify categorical columns (may have changed after transformations)
current_cat_cols = Data.select_dtypes(include=['category', 'object']).columns.tolist()

print(f"Categorical columns to encode: {current_cat_cols}")

for col in current_cat_cols:
    if col not in Data.columns:
        continue

    # Clean: Replace NaN strings with "Missing"
    Data[col] = Data[col].astype(str).replace({'nan': 'Missing', 'None': 'Missing', 'NaN': 'Missing'})
    Data[col] = Data[col].fillna('Missing')

    unique_count = Data[col].nunique()

    if unique_count <= 2:
        # LOW CARDINALITY (≤2): One-hot encoding
        # drop_first=True prevents the "dummy variable trap" (multicollinearity)
        # where one dummy can be predicted from the others
        Data = pd.get_dummies(Data, columns=[col], drop_first=True, dtype=float)
        encoding_log.append({'Column': col, 'Method': 'One-Hot', 'Unique': unique_count, 
                            'Reason': 'Low cardinality (≤2), drop_first prevents collinearity'})
        print(f"  ✅ {col}: One-hot encoded (drop_first=True), {unique_count} categories")

    elif unique_count < 5:
        # MEDIUM CARDINALITY (3-4): Label encoding
        # Maps categories to integers [0, 1, 2, ...]
        # Efficient and preserves order if categories are ordinal
        Data[col] = Data[col].astype('category').cat.codes
        encoding_log.append({'Column': col, 'Method': 'Label', 'Unique': unique_count,
                            'Reason': 'Medium cardinality (3-4), efficient integer encoding'})
        print(f"  ✅ {col}: Label encoded, {unique_count} categories → integers")

    else:
        # HIGH CARDINALITY (≥5): Semantic embeddings
        # We'll handle this in the next cell with Sentence-BERT
        encoding_log.append({'Column': col, 'Method': 'Semantic_Embedding', 'Unique': unique_count,
                            'Reason': 'High cardinality, semantic meaning captured via embeddings'})
        print(f"  ⏳ {col}: High cardinality ({unique_count}) → Semantic embedding (next cell)")

print(f"
{'─'*70}")
print("ENCODING SUMMARY:")
print(f"{'─'*70}")
enc_df = pd.DataFrame(encoding_log)
print(enc_df.to_string(index=False))



In [ ]:
# CELL 23: SEMANTIC EMBEDDINGS (Sentence-BERT)
# ═══════════════════════════════════════════════════════════════════════════════

print(f"
{'═'*70}")
print("FEATURE ENGINEERING: PHASE 5b - SEMANTIC EMBEDDINGS")
print(f"{'═'*70}")

# Identify columns that need semantic encoding
semantic_cols = [e['Column'] for e in encoding_log if e['Method'] == 'Semantic_Embedding']

if semantic_cols:
    print(f"Loading Sentence-BERT model for semantic encoding...")
    print(f"Model: 'all-MiniLM-L6-v2' (384-dimensional embeddings)")
    print(f"This model is pre-trained on 1+ billion sentence pairs and captures semantic meaning.")

    encoder_model = SentenceTransformer('all-MiniLM-L6-v2')

    for col in semantic_cols:
        if col not in Data.columns:
            continue

        # Get unique categories
        unique_cats = Data[col].unique().tolist()
        print(f"
  Encoding '{col}': {len(unique_cats)} unique categories")
        print(f"    Examples: {unique_cats[:5]}")

        # Generate semantic vectors (384 dimensions each)
        semantic_vectors = encoder_model.encode([str(c) for c in unique_cats])

        # Convert to PyTorch tensor for neural network compatibility
        pretrained_weights = t.FloatTensor(semantic_vectors)
        print(f"    Embedding shape: {pretrained_weights.shape}")

        # Create mapping: category → index
        cat_to_idx = {cat: idx for idx, cat in enumerate(unique_cats)}

        # Map data to indices
        Data[col] = Data[col].map(cat_to_idx)
        Data[col] = Data[col].astype(float)

        print(f"    ✅ '{col}' encoded as semantic indices (embedding matrix ready for NN)")

    print(f"
✅ Semantic encoding complete. Embedding matrices ready for neural network injection.")
else:
    print("  No high-cardinality columns require semantic encoding.")

# ─── Final type cleanup ───
# Convert all integer columns to float (neural networks prefer float)
int_cols = Data.select_dtypes(include="integer").columns
Data[int_cols] = Data[int_cols].astype(float)

# Convert remaining object columns to category
obj_cols = Data.select_dtypes(include="object").columns
Data[obj_cols] = Data[obj_cols].astype("category")

print(f"
✅ Final type cleanup complete.")
print(f"   Data shape: {Data.shape}")
print(f"   Data types: {Data.dtypes.value_counts().to_dict()}")



---

# Phase 5: Post-Feature Engineering Analysis

## Why Analyze Again?
After feature engineering, the data has changed fundamentally:
- Missing values are imputed
- Outliers are capped
- Distributions are transformed
- Features are scaled
- Categoricals are encoded

We must **verify** that:
1. Skewness is reduced (|skew| < 0.5)
2. No missing values remain
3. No infinite values exist
4. Distributions are reasonable
5. Correlations with target are preserved or improved

This is the **validation phase** — ensuring our engineering didn't break anything.



In [ ]:
# CELL 24: POST-FE VALIDATION
# ═══════════════════════════════════════════════════════════════════════════════

print(f"
{'═'*70}")
print("POST-FEATURE ENGINEERING VALIDATION")
print(f"{'═'*70}")

# ─── Check 1: No missing values ───
missing_post = Data.isnull().sum().sum()
print(f"
1️⃣  Missing Values: {missing_post}", end="")
print(" ✅ PASS" if missing_post == 0 else " ❌ FAIL")

# ─── Check 2: No infinite values ───
inf_count = np.isinf(Data.select_dtypes(include=[np.number])).sum().sum()
print(f"2️⃣  Infinite Values: {inf_count}", end="")
print(" ✅ PASS" if inf_count == 0 else " ❌ FAIL")

# ─── Check 3: No NaN after transforms ───
nan_count = Data.isna().sum().sum()
print(f"3️⃣  NaN Values: {nan_count}", end="")
print(" ✅ PASS" if nan_count == 0 else " ❌ FAIL")

# ─── Check 4: Skewness validation ───
numeric_post = Data.select_dtypes(include=[np.number]).columns
skew_violations = []
for col in numeric_post:
    s = Data[col].skew()
    if abs(s) > 1.0:
        skew_violations.append((col, s))

print(f"4️⃣  Skewness Violations (|skew| > 1.0): {len(skew_violations)}", end="")
print(" ✅ PASS" if len(skew_violations) == 0 else " ⚠️  WARNING")
if skew_violations:
    for col, s in skew_violations:
        print(f"     {col}: skew = {s:.3f}")

# ─── Check 5: Distribution summary ───
print(f"
{'─'*70}")
print("POST-FE DISTRIBUTION SUMMARY:")
print(f"{'─'*70}")
post_summary = Data[numeric_post].describe().T
print(post_summary[['mean', 'std', 'min', '50%', 'max']].round(4))

# ─── Check 6: Correlation preservation ───
print(f"
{'─'*70}")
print("POST-FE CORRELATION MATRIX (Sample):")
print(f"{'─'*70}")
if len(numeric_post) > 1:
    post_corr = Data[numeric_post].corr()
    # Show only upper triangle for readability
    mask = np.triu(np.ones_like(post_corr, dtype=bool), k=1)
    strong_post = []
    for i in range(len(post_corr.columns)):
        for j in range(i+1, len(post_corr.columns)):
            r = post_corr.iloc[i, j]
            if abs(r) > 0.7:
                strong_post.append((post_corr.columns[i], post_corr.columns[j], r))

    if strong_post:
        print("  Strong correlations (|r| > 0.7):")
        for a, b, r in strong_post:
            print(f"    {a} ↔ {b}: r = {r:.3f}")
    else:
        print("  ✅ No severe multicollinearity detected post-transformation")

print(f"
{'═'*70}")
print("✅ POST-FE VALIDATION COMPLETE")
print(f"{'═'*70}")
print("The dataset is now clean, transformed, scaled, and encoded.")
print("Ready for model training.")



---

# Appendix: Neural Network Architecture (SemanticTabularNet)

This architecture leverages the semantic embeddings we created:

```python
class SemanticTabularNet(nn.Module):
    def __init__(self, pretrained_weights, num_continuous_features):
        super().__init__()
        
        # 1. Semantic Embedding Layer
        # freeze=True: Don't overwrite pre-trained semantic knowledge during training
        self.occupation_embed = nn.Embedding.from_pretrained(pretrained_weights, freeze=True)
        
        embed_dim = pretrained_weights.shape[1]  # 384 for all-MiniLM-L6-v2
        
        # 2. Feature Fusion: Concatenate semantic + continuous features
        total_input_size = embed_dim + num_continuous_features
        
        # 3. Dense Backbone
        self.fc1 = nn.Linear(total_input_size, 256)
        self.mish = nn.Mish()  # Smoother than ReLU, avoids dead neurons
        self.dropout = nn.Dropout(0.3)  # Regularization
        self.out = nn.Linear(256, 1)  # Output layer
        
    def forward(self, cat_idx, num_features):
        # Look up semantic vectors
        x_cat = self.occupation_embed(cat_idx)
        
        # Concatenate with numeric features
        x = torch.cat([x_cat, num_features], dim=1)
        
        # Forward through dense layers
        x = self.dropout(self.mish(self.fc1(x)))
        return self.out(x)
```

**Why this architecture works:**
1. **Frozen embeddings** preserve world knowledge from pre-training
2. **Mish activation** is smoother than ReLU, improving gradient flow
3. **Dropout** prevents overfitting on tabular data
4. **Concatenation** allows the network to learn interactions between semantic and numeric features



In [ ]:
{'🎉'*20}")
print("EDA PIPELINE COMPLETE")
print(f"{'🎉'*20}")
print("
All visualizations saved to /mnt/agents/output/")
print("All transformations documented with rationale.")
print("Dataset is model-ready.")
